# 01 - NYC Demand / Congestion / Quantile ETA training (Colab)

Thin runner notebook. **No training logic lives here** - every cell below calls straight into the existing, already-tested modules under `models/xgboost_model/`, `models/congestion/`, `models/eta/`, `models/data_prep/`. If a cell looks like it's doing feature engineering or model fitting, that's a bug - file it against this notebook, not against the `.py` modules.

What this trains:
1. **NYC zone-hourly demand** (`models/xgboost_model/train_xgboost.py`) - reads the pre-aggregated `zone_hourly_demand` mart (1.44M rows total, already small; see the sampling note in the parameters cell for why this one does *not* take a `--sample-size`).
2. **Congestion multiplier** (`models/congestion/train_congestion_xgb.py`) - reservoir-samples `int_trips_enriched` (113M raw rows) via a DuckDB `USING SAMPLE ... ROWS (reservoir, seed)` pushdown, never loads the full table into pandas. This is where progressive sampling (1M-5M-10M-25M-50M-ALL) actually matters.
3. **Quantile ETA (p10/p50/p90)** (`models/eta/train_quantile_eta.py`) - same reservoir-sampled feature table as congestion (reused, not rebuilt).

Progressive-sampling honesty note: `build_features(con, sample_rows=N)` in both `models/congestion/build_features.py` and (indirectly, same function) `models/eta/train_quantile_eta.py` **already accepted a `sample_rows` parameter with DuckDB-side reservoir sampling before this notebook was written** - nothing was added to those two files. The NYC demand model (`models/xgboost_model/train_xgboost.py`) has no such parameter and none was added: its source table is already a 1.44M-row aggregated mart, not the 113M-row raw table, so a row-sample knob wouldn't reduce compute in any meaningful way - see the parameters cell below.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Locate the repo and the data

Two things are needed and they live in different places:
- **Code**: this repo (small, git-friendly) - cloned fresh each session.
- **Data**: `data/warehouse/nyc_rides.duckdb` is **6.0 GB** (verified locally, `IMPLEMENTATION_AUDIT.md` section 2) - too large to commit to git, so it is *not* part of the clone. It must already exist somewhere reachable from Colab: Google Drive (mounted above) or a GCS bucket.

Set `GLOBAL_MOBILITY_DATA_ROOT` to wherever you put it - this notebook never hardcodes one person's Drive folder layout. Two supported layouts, pick one:
- **Drive**: upload `nyc_rides.duckdb` (and `london_cycles.duckdb` if you also run notebook 02) into a Drive folder, then set `GLOBAL_MOBILITY_DATA_ROOT=/content/drive/MyDrive/<your-folder>`.
- **GCS**: `gsutil cp gs://<bucket>/nyc_rides.duckdb /content/data/` first, then set `GLOBAL_MOBILITY_DATA_ROOT=/content/data`.

In [ ]:
import os

REPO_URL = os.environ.get('GLOBAL_MOBILITY_REPO_URL', 'https://github.com/TeerthPurohit/Uber-nyc-TLC-Dataset.git')
REPO_DIR = '/content/nyc-tlc-repo'

# GLOBAL_MOBILITY_DATA_ROOT: folder containing nyc_rides.duckdb (and
# london_cycles.duckdb). Defaults to this user's actual Drive upload location
# (My Drive/global_mobility/repo) - override via env var if you move it.
os.environ.setdefault('GLOBAL_MOBILITY_DATA_ROOT', '/content/drive/MyDrive/global_mobility/repo')
DATA_ROOT = os.environ['GLOBAL_MOBILITY_DATA_ROOT']

if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" pull

print('repo:', REPO_DIR)
print('data root:', DATA_ROOT)

## 3. Install pinned dependencies (parity with local `requirements.txt`)

In [ ]:
# Reuse the repo's own pin file instead of re-declaring versions here -
# this is the same requirements.txt models/*/train_*.py are developed
# against locally.
!pip install -q -r "$REPO_DIR/requirements.txt"

## 4. Wire the DuckDB file into the expected repo path

`models/xgboost_model/train_xgboost.py` / `models/congestion/train_congestion_xgb.py` / `models/eta/train_quantile_eta.py` all default to `DEFAULT_DB_PATH = <repo_root>/data/warehouse/nyc_rides.duckdb` (hardcoded relative to each module's own file location, not overridable via env var - verified in each script). Rather than patch those constants, symlink the real (large) file from `GLOBAL_MOBILITY_DATA_ROOT` into that exact path.

In [ ]:
import shutil
warehouse_dir = os.path.join(REPO_DIR, 'data', 'warehouse')
os.makedirs(warehouse_dir, exist_ok=True)

src = os.path.join(DATA_ROOT, 'nyc_rides.duckdb')
dst = os.path.join(warehouse_dir, 'nyc_rides.duckdb')
assert os.path.exists(src), (
    f'nyc_rides.duckdb not found at {src} -- set GLOBAL_MOBILITY_DATA_ROOT to '
    'the Drive/GCS folder that actually contains it before continuing.'
)
if not os.path.exists(dst):
    os.symlink(src, dst)
print('duckdb file ready at', dst, f'({os.path.getsize(dst) / 1e9:.2f} GB)')

## 5. Validate schema and row counts before training (fail loudly, not blind)

Checks the exact tables/columns `models/data_prep/build_features.py` and `models/congestion/build_features.py` query, per `IMPLEMENTATION_AUDIT.md` section 3 - if these don't match, every downstream training cell would either crash confusingly or silently train on the wrong shape of data.

In [ ]:
import duckdb

EXPECTED = {
    'zone_hourly_demand': {'pickup_date', 'pickup_hour', 'pickup_location_id', 'total_trips', 'temperature_c', 'precipitation_mm'},
    'int_trips_enriched': {'pickup_at', 'pickup_location_id', 'trip_distance', 'trip_duration_minutes', 'avg_speed_mph'},
}
MIN_ROWS = {'zone_hourly_demand': 1_000_000, 'int_trips_enriched': 50_000_000}

con = duckdb.connect(dst, read_only=True)
try:
    tables = {r[0] for r in con.execute("select table_name from information_schema.tables").fetchall()}
    for table, expected_cols in EXPECTED.items():
        if table not in tables:
            raise RuntimeError(f'expected table {table!r} not found in {dst} -- wrong/stale duckdb file?')
        cols = {r[0] for r in con.execute(f'describe {table}').fetchall()}
        missing = expected_cols - cols
        if missing:
            raise RuntimeError(f'{table} is missing expected columns {missing} -- schema drift, do not train blind')
        n = con.execute(f'select count(*) from {table}').fetchone()[0]
        if n < MIN_ROWS[table]:
            raise RuntimeError(f'{table} has only {n} rows, expected >= {MIN_ROWS[table]} -- looks like a partial/sample file')
        print(f'{table}: OK, {n:,} rows, columns match')
finally:
    con.close()

## 6. Progressive sampling parameter

Applies to **congestion** and **ETA** only (they read from the 113M-row `int_trips_enriched`). Start small, only scale up if held-out metrics meaningfully improve - see `models/notebooks/README.md`. `None` means "use every row" (`ALL`, no `USING SAMPLE` clause at all - see `load_raw_trips()` in `models/congestion/build_features.py`).

In [ ]:
# Bump this across runs: 1_000_000 -> 5_000_000 -> 10_000_000 -> 25_000_000
# -> 50_000_000 -> None (ALL ~113M rows). Re-run section 7/8 after changing it.
SAMPLE_ROWS = 1_000_000
print('training congestion + ETA on sample_rows =', SAMPLE_ROWS if SAMPLE_ROWS is not None else 'ALL (~113M rows)')

## 7. Train: NYC zone-hourly demand (XGBoost)

Calls `models/xgboost_model/train_xgboost.py`'s `train_and_save()` directly - no CLI subprocess needed, it's already an importable function. Saves `xgb_model.json` + `xgb_metadata.json` + `feature_importance.png` into `models/xgboost_model/` inside the cloned repo, same filenames `backend/services/model_service.py` and `scripts/refresh_model_registry.py` already expect.

In [ ]:
import sys, time
sys.path.insert(0, REPO_DIR)
from models.xgboost_model.train_xgboost import train_and_save as train_demand, plot_feature_importance

t0 = time.perf_counter()
demand_meta = train_demand()
demand_elapsed_s = time.perf_counter() - t0
plot_feature_importance(demand_meta, f'{REPO_DIR}/models/xgboost_model/feature_importance.png')

print(f"n_rows: {demand_meta['n_rows']}")
print(f"val   RMSE={demand_meta['metrics']['val_rmse']:.3f}  MAE={demand_meta['metrics']['val_mae']:.3f}")
print(f"test  RMSE={demand_meta['metrics']['test_rmse']:.3f}  MAE={demand_meta['metrics']['test_mae']:.3f}")
print(f'training time: {demand_elapsed_s:.1f}s')

## 8. Train: congestion multiplier + quantile ETA, at the sample size set above

Both call into `models/congestion/build_features.py`'s reservoir-sampled query via each script's own `train_and_save(sample_rows=...)` - this is the actual DuckDB-side `USING SAMPLE N ROWS (reservoir, seed)` predicate pushdown (`models/congestion/build_features.py::load_raw_trips`), never `pd.read_parquet().sample()`.

In [ ]:
from models.congestion.train_congestion_xgb import train_and_save as train_congestion
from models.eta.train_quantile_eta import train_and_save as train_eta

run_log = []

t0 = time.perf_counter()
congestion_meta = train_congestion(sample_rows=SAMPLE_ROWS)
congestion_elapsed_s = time.perf_counter() - t0
run_log.append({
    'model': 'congestion', 'sample_rows': SAMPLE_ROWS, 'n_rows_test': congestion_meta['n_rows']['test'],
    'test_rmse': congestion_meta['metrics']['test_rmse'], 'test_mae': congestion_meta['metrics']['test_mae'],
    'train_time_s': congestion_elapsed_s,
})
print('congestion:', run_log[-1])

t0 = time.perf_counter()
eta_meta = train_eta(sample_rows=SAMPLE_ROWS)
eta_elapsed_s = time.perf_counter() - t0
run_log.append({
    'model': 'eta', 'sample_rows': SAMPLE_ROWS, 'n_rows_test': eta_meta['n_rows']['test'],
    'p50_mae': eta_meta['metrics']['p50']['mae'],
    'p10_p90_coverage': eta_meta['prediction_interval_coverage']['measured_p10_p90_coverage'],
    'train_time_s': eta_elapsed_s,
})
print('eta:', run_log[-1])

## 9. Record this run (rows / time / metrics) - repeat sections 6-9 per sample size

Re-run cell 6 with a bigger `SAMPLE_ROWS`, then re-run cell 8, then re-run this cell - `run_log` accumulates every sample size actually run in this session so you can see whether a bigger sample was worth the extra training time before committing to `ALL`.

In [ ]:
import pandas as pd
pd.DataFrame(run_log)

## 10. Copy artifacts back to the local repo

After a training run you're happy with, copy these files from the Colab filesystem back into your local clone at the **same relative paths** - `scripts/refresh_model_registry.py` and `backend/services/model_service.py` both key off these exact filenames, nothing else needs to change.

In [ ]:
import shutil

OUTPUT_DIR = f'{DATA_ROOT}/trained_artifacts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILES_TO_COPY = [
    'models/xgboost_model/xgb_model.json',
    'models/xgboost_model/xgb_metadata.json',
    'models/xgboost_model/feature_importance.png',
    'models/congestion/congestion_model.json',
    'models/congestion/congestion_metadata.json',
    'models/eta/eta_p10_model.json',
    'models/eta/eta_p50_model.json',
    'models/eta/eta_p90_model.json',
    'models/eta/eta_metadata.json',
]
for rel in FILES_TO_COPY:
    src = os.path.join(REPO_DIR, rel)
    dst_path = os.path.join(OUTPUT_DIR, rel.replace('/', '__'))
    shutil.copy(src, dst_path)

print(f'Copied {len(FILES_TO_COPY)} files to {OUTPUT_DIR}.')
print('Next: download these from Drive and place each one back at its original')
print('relative path under your local repo checkout, e.g.:')
for rel in FILES_TO_COPY:
    print(f'  {OUTPUT_DIR}/{rel.replace(chr(47), chr(95)+chr(95))}  ->  <local repo>/{rel}')
print()
print('Then locally: python scripts/refresh_model_registry.py')